## 25. TypeScript 专属能力对照

> 来源：[Agent SDK reference - TypeScript](https://code.claude.com/docs/en/agent-sdk/typescript)、[TypeScript SDK V2 session API (removed)](https://code.claude.com/docs/en/agent-sdk/typescript-v2-preview)

本笔记以 Python 为主线，这一章集中回答两个问题：**选型时**哪些能力只在 TypeScript SDK 存在；**读 TS 示例或文档时**同一能力在两边分别长什么样。与 Python 完全对等的部分（权限规则语义、setting_sources、session 管理函数、file checkpointing、skills/plugins、MCP 三种 transport、超时环境变量等）不重复展开，正文各章已覆盖。

### 25.1 影响选型的差异（必须知道）

- **`env` 语义相反**：TS 传 `env` 是**整体替换**子进程环境——必须写 `{ ...process.env, YOUR_VAR: "v" }`，否则连 PATH 都会丢；Python 是**叠加合并**，直接传增量 dict 即可（§8「配置总线」④组）。
- **`persistSession: false` 仅 TS 有**：关闭会话落盘（之后不能 resume），服务端多租户/无状态部署的关键开关；Python 始终写盘，临时化只能靠把 `CLAUDE_CONFIG_DIR` 指到临时目录（§13.3「session_store」）。
- **`permissionMode: "auto"` 仅 TS 有**：用模型分类器逐个判定 tool call 批还是拒；Python 的 `PermissionMode` 只有 default / acceptEdits / plan / dontAsk / bypassPermissions 五档（§9.2「permission_mode 全表」）。
- **`Workflow` tool 仅 TS 有**（SDK ≥ 0.3.149）：一段脚本在后台确定性编排多个子 agent（`agent()` / `parallel()` / `pipeline()` / `phase()`），支持 `resumeFromRunId` 断点续跑；坑点是输出 `status` 恒为 `"async_launched"`，**必须检查 `error` 字段**——脚本语法错误也照样返回 async_launched。
- **10 个 hook 事件仅 TS 有**：`SessionStart` / `SessionEnd` / `Setup` / `TeammateIdle` / `TaskCompleted` / `ConfigChange` / `WorktreeCreate` / `WorktreeRemove` / `PostToolBatch`（一批 tool call 全部落定后、下一次模型请求前触发一次）/ `MessageDisplay`（逐 delta 渲染事件）。Python 侧的替代出路见 §11.1「Python 可用的 hook 事件」。
- **sandbox `failIfUnavailable` 默认值相反**：TS 默认 `true`——沙箱起不来直接报 `error_during_execution`；Python 默认回退为**无沙箱执行**、只在 stderr 打警告（字段没在 `SandboxSettings` 上声明，但手写 `"failIfUnavailable": True` 会被透传生效，§21.5「Sandbox」）。安全敏感部署在 Python 侧必须显式设置。
- **`applyFlagSettings()` 仅 TS 有**（官方明说 Python 无对等方法）：运行中把任意 settings 子集写入 flag 层（覆盖 user/project/local，仅 managed policy 更高），`model`/`permissions`/`hooks` 等下一 turn 生效；传 `null` 清除某 key。
- **`setting_sources=[]` 的版本坑只在 Python**：SDK ≤ 0.1.59 把空列表当省略处理（等于全加载），TS 不受影响（§21.8「迁移」）。
- **`canUseTool` 被遮蔽告警仅 TS 有**（v2.1.198 起，来自 permissions 文档）：回调因 `bypassPermissions` 或裸名 allow 规则而永远不可能被调用时，发一次 Node 进程告警 `CLAUDE_SDK_CAN_USE_TOOL_SHADOWED`；Python 侧没有告警，同样的遮蔽只能自己对照 §9.1「权限评估顺序」排查。

### 25.2 TS 专属能力全景

**生命周期与运行环境**

| 能力 | 说明 |
|---|---|
| `startup()` / `WarmQuery` | 预热 CLI 子进程（提前 spawn + initialize 握手），之后 `warm.query(prompt)` 零启动延迟；`WarmQuery` 只能 query 一次 |
| `resolveSettings()`（alpha） | 不 spawn CLI，用与 CLI 相同的合并引擎解析某目录的有效 settings，返回逐 key 的 `provenance`（来源层） |
| `spawnClaudeCodeProcess` | 自定义 spawn 函数，把 CLI 跑进 VM/容器/远端——只接管进程 spawn，stdio 协议仍由 SDK 驱动（Python 对应物是接管整个通信通道的 `Transport`） |
| `executable` / `executableArgs`、`debug` / `debugFile` | 选择 JS 运行时（bun/deno/node）；CLI debug 日志落文件 |
| Bun 单文件打包 | `bun build --compile` 后需 `extractFromBunfs()`（v0.3.144+）把捆绑的原生 CLI 二进制解出再传路径 |

**`Query` 对象的控制面**（Python `ClaudeSDKClient` 没有的方法）

| 方法 | 说明 |
|---|---|
| `applyFlagSettings()` | 见 §25.1 |
| `initializationResult()` / `reinitialize()` | 初始化结果查询；传输中断重连后重发 initialize，把 pending 权限请求重投给 `canUseTool`（需按 request ID 幂等） |
| `supportedCommands/Models/Agents()`、`accountInfo()` | 类型化返回命令/模型（含 `supportsEffort` 等能力位）/agent/账号信息；Python 只有粗粒度 `get_server_info()` |
| `setMcpServers()` | 会话中整体替换 MCP server 集合，返回 `{added, removed, errors}`；Python 只能逐个 reconnect/toggle |
| `streamInput()` / `close()` | 向进行中的 query 追加输入流；强制关闭并终止底层进程 |

**Options 专属字段**（Python `ClaudeAgentOptions` 没有的）

| 字段 | 说明 |
|---|---|
| `persistSession` | 见 §25.1 |
| `allowDangerouslySkipPermissions` | 用 `bypassPermissions` 时必须同时置 `true` 的双保险；Python 没有这道栓 |
| `agent` | 指定主线程直接以某个已定义 agent 的身份运行 |
| `toolAliases` | 把内置工具名映射到 MCP 工具（如 `{Bash: "mcp__workspace__bash"}`），用自己的实现替换内置工具 |
| `forwardSubagentText` | 把子 agent 的 text/thinking 块带 `parent_tool_use_id` 转发，可渲染嵌套 transcript（默认只转发 tool_use/tool_result） |
| `onElicitation` | MCP server 请求用户输入（elicitation）时的回调，不提供则自动拒绝 |
| `planModeInstructions` | 替换 plan 模式的默认工作流正文 |
| `resumeSessionAt` / `sessionId` / `title` | 从指定消息 UUID 恢复 / 指定会话 UUID / 设置会话标题 |
| `settings` 可传内联对象 | TS 接受 `string \| Settings`；Python 只接受文件路径 |
| `agentProgressSummaries` / `promptSuggestions` / `managedSettings` / `taskBudget`（alpha）/ `toolConfig.askUserQuestion.previewFormat` / `abortController` | 子 agent 进度一行摘要 / 预测下一条 prompt / 宿主注入策略层 / API 侧 token 预算 / `AskUserQuestion` 选项带视觉预览 / 取消控制 |

**消息流与诊断**

| 能力 | 说明 |
|---|---|
| 结果消息诊断字段 | `ttft_ms`（首个完整 assistant 消息耗时）、`ttft_stream_ms`（首个流事件耗时）、`terminal_reason`（12 种终止原因枚举）、`fast_mode_state`；Python `ResultMessage` 均无 |
| `SDKUserMessage.shouldQuery: false` | 消息只追加进 transcript、不触发 assistant turn——免费注入带外上下文 |
| `SDKMessageOrigin` | user 消息与 result 带来源标签（human / channel / peer / task-notification / coordinator / auto-continuation），可过滤合成 turn |
| 约 30 种强类型消息 | `permission_denied`（含拒绝原因分类）、`task_updated`、`commands_changed`、`tool_progress`、`prompt_suggestion` 等都是独立类型；Python 把 system 事件统一落进 `SystemMessage(subtype, data)` |
| `canUseTool` 的 `requestId` + 返回 `null` | 应用可自行在 SDK 通道外回 `control_response` 再返回 `null` 跳过写回（其他情况返回 null 会让 tool call 永久卡死） |
| hook 输入更丰富 | `BaseHookInput` 带 `prompt_id`（可关联 OTel 事件）、`effort`；`Stop`/`SubagentStop` 输入带 `last_assistant_message`、`background_tasks` |
| `deferred_tool_use` 闭环 | hook `defer` 后 result 带 `stop_reason: "tool_deferred"` 和 `deferred_tool_use: {id, name, input}`，自建审批 UI 后同 session resume 续跑 |

**工具与沙箱**

| 能力 | 说明 |
|---|---|
| `Workflow` tool | 见 §25.1 |
| `structuredContent` 返回 | 自定义工具结果可带机器可读 JSON（§10.7「structuredContent」——Python 进程内 server 不支持的正是这个 TS 能力） |
| `SandboxSettings.filesystem` / `ripgrep` | 文件系统读写黑白名单（`allowWrite`/`denyWrite`/`denyRead`）与自定义 ripgrep 二进制；Python 的 `SandboxSettings` 无这两项 |
| `McpClaudeAIProxyServerConfig` | MCP server 配置多一种 `type: "claudeai-proxy"`；`PermissionUpdateDestination` 多 `"cliArg"` 档 |
| `AgentDefinition.criticalSystemReminder_EXPERIMENTAL` | 实验性字段，Python 无 |

### 25.3 同能力不同形态（读 TS 代码时对号入座）

| 主题 | TS 形态 | Python 形态 |
|---|---|---|
| 交互总纲 | 单一 `query()` 返回 `Query` 对象——AsyncGenerator 和全部控制方法挂在同一对象上；流式输入=传 `AsyncIterable<SDKUserMessage>` | 双轨：`query()` 一次性迭代器（无控制方法）+ `ClaudeSDKClient` 承担持续会话与控制面（§4「两个入口」） |
| `can_use_tool` 前提 | 字符串 prompt 即可用 | 需要流式输入 + dummy `PreToolUse` hook 保持控制流打开（§9.3「can_use_tool 回调」） |
| 自定义工具定义 | `tool(name, desc, zodRawShape, handler, {annotations})`，Zod schema，入参自动带类型 | `@tool(name, desc, schema, annotations=)` 装饰器，schema 用简易 dict 或 JSON Schema（§10.2「Schema 两档写法」） |
| 回调上下文 | `canUseTool` 第三参含可用的 `signal`（AbortSignal）、`toolUseID`、`agentID`、`requestId` | `ToolPermissionContext`：`signal` 仅占位；但多了 TS 没有的 UI 文案字段 `title` / `display_name` / `description`（§9.3） |
| 权限结果 | 匿名判别联合 `{behavior: "allow", ...} \| {behavior: "deny", ...}` | 具名 dataclass `PermissionResultAllow` / `PermissionResultDeny` |
| hook 返回字段 | `continue` / `async` 直接写；`decision` 有 `"approve"` 和 `"block"` 两值 | 保留字加下划线 `continue_` / `async_`（发送时自动转换）；`decision` 只有 `"block"`（§11.3「回调签名与返回值」） |
| 消息对象 | `SDKAssistantMessage.message` 是 Anthropic SDK 原生 `BetaMessage`，content block 要自己过滤；约 30 种消息全部强类型 | 解析成 dataclass，`AssistantMessage.content` 直接是 ContentBlock 列表按 `isinstance` 分派；system 事件落泛型 `SystemMessage(subtype, data)`（§7「读懂消息流」） |
| 结果消息 | `SDKResultMessage` 是 success/error 两臂判别联合，success 臂才有 `result` / `ttft_ms` / `structured_output` | `ResultMessage` 单一 dataclass 摊平全部变体，不适用字段为 `None`，靠 `subtype` 判断 |
| 取消/中断 | `options.abortController` + `AbortError`；`query.interrupt()` / `query.close()` | 无 abort 原语：`client.interrupt()`（仅流式）、`client.disconnect()` 或取消 asyncio 任务（§15「多轮与打断」） |
| 错误呈现 | 无异常层级，错误以消息内字段呈现（error subtype、`api_error_status`）+ `AbortError` | 异常层级 `ClaudeSDKError` → `CLIConnectionError` → `CLINotFoundError`，另有 `ProcessError` / `CLIJSONDecodeError`（§21.1「错误处理」） |
| 远端运行 | `spawnClaudeCodeProcess` 只接管进程 spawn | `Transport` ABC 接管整个通信通道（低层内部 API，接口可能变） |
| 命名约定 | 全 camelCase（`pathToClaudeCodeExecutable`、`maxTurns`、`continue`） | 全 snake_case（`cli_path`、`max_turns`、`continue_conversation`）——**唯一例外是 `AgentDefinition` 保持 camelCase**（§14.1「编程式定义」的命名坑） |
| `TodoWrite` 停用方式 | TS SDK ≥ 0.3.142 默认停用、切 Task 系工具 | 跟随 CLI 版本 ≥ v2.1.142，与 Python 包版本无关（§24「Todo 跟踪」） |

### 25.4 V2 session API 的兴衰（已移除）

V2 是 TS 侧的一次实验：把 `query()` 单一 async generator 的"输入输出挤在一条流里"拆开成显式的三概念——`createSession()` / `resumeSession()` 开始或继续会话、`session.send()` 发消息、`session.stream()` 取响应，多轮之间可以随意插处理逻辑。

结局：**TS SDK 0.3.142 把 `unstable_v2_createSession` / `unstable_v2_resumeSession` / `unstable_v2_prompt` 及相关类型整体移除**（版本号从 0.2.x 直接跳到 0.3.142，两者是同一边界）；官方迁移路径是回到 V1 `query()`——多轮用 `AsyncIterable<SDKUserMessage>` 输入流，续会话用 `options.resume`。文档页仅为仍锁在 `@0.2` 的代码保留。

对 Python 读者，这段兴衰有一个对照价值：V2 的 send/stream 分离形态，正是 Python `ClaudeSDKClient.query()` + `receive_response()` 的形态——可以理解为 TS 曾试图长出一套 `ClaudeSDKClient` 风格的 API，后又砍掉、回归单一 `query()`。两个 SDK 在"持续会话怎么建模"上最终走了相反的路（§4「两个入口」、§15「多轮与打断」）。